<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/ewpd4lhc_wilson_ray_complete_colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EWPD4LHC → Wilson-Ray Inputs (Flavor-Universal Default) + χ²⊥ Test (Robust YAML Path Discovery)

This Colab/Jupyter notebook:

1. Clones `ewpd4lhc/ewpd4lhc`.
2. Runs the **default** flavor-universal build to generate a YAML output (typically `ewpd_out.yml`).
3. **Discovers** where the coefficient names and numeric matrices live inside the YAML by recursively scanning keypaths.
4. Extracts:
   - `coeff_names` (operator ordering),
   - `A` (observable response/Jacobian),
   - `V` or `Vinv` (observable covariance or precision),
   - builds the coefficient-space Fisher matrix `F = Aᵀ V⁻¹ A`,
   - computes `SigmaC = pinv(F)` and rank diagnostics.
5. Computes χ²⊥ once you paste your ray vector `v_ray` in the extracted ordering.

This is a one-time extraction + linear algebra evaluation.

## 0) Environment check

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

## 1) Clone repository

In [ ]:
!git clone https://github.com/ewpd4lhc/ewpd4lhc.git
%cd ewpd4lhc
!ls -la

## 2) Install dependencies

In [ ]:
!pip -q install numpy pyyaml scipy pandas

## 3) Run default EWPD build (flavor-universal)

Expected output is a YAML file (often `ewpd_out.yml`). If the filename differs,
the directory listing will show it.

In [ ]:
!chmod +x ewpd4lhc.py
!./ewpd4lhc.py
!ls -lh

## 4) Load YAML output

In [ ]:
import yaml
from pathlib import Path

candidates = ["ewpd_out.yml", "ewpd_out.yaml", "out.yml", "out.yaml"]
yml_path = None
for c in candidates:
    p = Path(c)
    if p.exists():
        yml_path = p
        break
if yml_path is None:
    ymls = sorted(list(Path(".").glob("*.yml")) + list(Path(".").glob("*.yaml")))
    if not ymls:
        raise FileNotFoundError("No .yml/.yaml output found after running ewpd4lhc.py.")
    yml_path = ymls[0]

print("Using YAML:", yml_path)

with open(yml_path, "r") as f:
    Y = yaml.safe_load(f)

print("Top-level type:", type(Y).__name__)
if isinstance(Y, dict):
    print("Top-level keys (first 120):")
    keys = list(Y.keys())
    print(keys[:120])

## 5) Recursive scan of YAML keypaths

This produces candidate keypaths for:
- string lists (often coefficient/operator names),
- numeric vectors,
- numeric matrices (often A, V, Vinv, Fisher, etc.).

In [ ]:
import numpy as np

def iter_paths(obj, prefix=()):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from iter_paths(v, prefix + (str(k),))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_paths(v, prefix + (f"[{i}]",))
    else:
        yield prefix, obj

def is_numeric_scalar(x):
    return isinstance(x, (int, float, np.integer, np.floating))

def list_is_str_list(L, min_len=5):
    return isinstance(L, list) and len(L) >= min_len and all(isinstance(x, str) for x in L)

def list_is_num_vector(L, min_len=5):
    return isinstance(L, list) and len(L) >= min_len and all(is_numeric_scalar(x) for x in L)

def list_is_num_matrix(L, min_shape=(5,5)):
    if not (isinstance(L, list) and len(L) >= min_shape[0]):
        return False
    if not all(isinstance(r, list) for r in L):
        return False
    ncol = len(L[0])
    if ncol < min_shape[1]:
        return False
    if not all(len(r) == ncol for r in L):
        return False
    return all(all(is_numeric_scalar(x) for x in r) for r in L)

str_lists = []
num_vectors = []
num_matrices = []

for path, val in iter_paths(Y):
    if list_is_str_list(val):
        str_lists.append((path, len(val)))
    elif list_is_num_vector(val):
        num_vectors.append((path, len(val)))
    elif list_is_num_matrix(val):
        nrow = len(val)
        ncol = len(val[0]) if nrow else 0
        num_matrices.append((path, (nrow, ncol)))

print("=== String-list candidates (possible coefficient/operator names) ===")
for p, n in sorted(str_lists, key=lambda x: -x[1])[:30]:
    print("  ", " / ".join(p), "   len=", n)

print("\n=== Numeric-vector candidates (possible best-fit / sigma vectors) ===")
for p, n in sorted(num_vectors, key=lambda x: -x[1])[:30]:
    print("  ", " / ".join(p), "   len=", n)

print("\n=== Numeric-matrix candidates (possible A, V, Vinv, etc.) ===")
for p, sh in sorted(num_matrices, key=lambda x: -(x[1][0]*x[1][1]))[:30]:
    print("  ", " / ".join(p), "   shape=", sh)

## 6) Select YAML keypaths and extract (`coeff_names`, `A`, `V`/`Vinv`)

Set the path tuples below based on the candidates printed in Section 5.

Rules:
- `coeff_names`: string list of length N.
- `A`: numeric matrix with shape (M, N).
- `V` or `Vinv`: numeric matrix with shape (M, M) where M = number of rows of `A`.

In [ ]:
import numpy as np

def get_by_path(root, path_tuple):
    obj = root
    for k in path_tuple:
        if k.startswith("[") and k.endswith("]"):
            idx = int(k[1:-1])
            obj = obj[idx]
        else:
            obj = obj[k]
    return obj

# -----------------------------
# REQUIRED MANUAL PATHS (edit)
# -----------------------------
PATH_COEFF_NAMES = None   # e.g. ("likelihood","wc_names")
PATH_A           = None   # e.g. ("likelihood","A")
PATH_V           = None   # e.g. ("likelihood","V")
PATH_VINV        = None   # alternative: set this instead of PATH_V

if PATH_COEFF_NAMES is None or PATH_A is None or (PATH_V is None and PATH_VINV is None):
    raise ValueError("Set PATH_COEFF_NAMES, PATH_A, and PATH_V (or PATH_VINV).")

coeff_names = list(get_by_path(Y, PATH_COEFF_NAMES))
A = np.array(get_by_path(Y, PATH_A), dtype=float)

if PATH_VINV is not None:
    Vinv = np.array(get_by_path(Y, PATH_VINV), dtype=float)
else:
    V = np.array(get_by_path(Y, PATH_V), dtype=float)
    Vinv = np.linalg.pinv(V)

print("len(coeff_names) =", len(coeff_names))
print("A shape          =", A.shape)
print("Vinv shape       =", Vinv.shape)

if A.shape[1] != len(coeff_names):
    raise ValueError("Mismatch: A.shape[1] must equal len(coeff_names). Wrong PATH_COEFF_NAMES or PATH_A.")
if Vinv.shape[0] != Vinv.shape[1] or Vinv.shape[0] != A.shape[0]:
    raise ValueError("Mismatch: Vinv must be (M,M) with M=A.shape[0]. Wrong PATH_V/PATH_VINV.")

## 7) Build Fisher matrix and rank diagnostics

\[
F = A^{T} V^{-1} A, \qquad \Sigma_C = F^{+}.
\]

In [ ]:
F = A.T @ Vinv @ A
SigmaC = np.linalg.pinv(F)

svals = np.linalg.svd(F, compute_uv=False)
tol = max(F.shape) * np.max(svals) * 1e-12
rankF = int(np.sum(svals > tol))

print("F shape:", F.shape)
print("rank(F):", rankF, "out of", F.shape[0])
print("Smallest singular values (last 10):", svals[-10:])

## 8) Best-fit / central vector `C_hat` (optional)

If your YAML includes a coefficient best-fit vector, set `PATH_C_HAT`.
Otherwise the default is the zero vector.

In [ ]:
PATH_C_HAT = None  # e.g. ("fit","C_hat") (optional)

import numpy as np
if PATH_C_HAT is None:
    C_hat = np.zeros((len(coeff_names), 1), dtype=float)
    print("Using C_hat = 0 (no PATH_C_HAT set).")
else:
    C_hat = np.array(get_by_path(Y, PATH_C_HAT), dtype=float).reshape(-1, 1)
    print("Loaded C_hat from PATH_C_HAT.")

if C_hat.shape[0] != len(coeff_names):
    raise ValueError("C_hat length mismatch with coeff_names. Wrong PATH_C_HAT or coeff list.")

## 9) Coefficient ordering (paste `v_ray` in this ordering)

In [ ]:
import pandas as pd
display(pd.DataFrame({"i": range(len(coeff_names)), "coef": coeff_names}).head(200))
print("Total coefficients:", len(coeff_names))

v_ray = None  # paste your ray vector here (length N)

## 10) Compute χ²⊥

\[
\chi^2_{\perp} = \hat C^{T} F \hat C - \frac{(v^{T} F \hat C)^2}{v^{T} F v}.
\]

Effective dof: \(\nu_{\mathrm{eff}} = \mathrm{rank}(F)-1\).

In [ ]:
def chi2_perp_from_F(C_hat, F, v_ray):
    C = np.asarray(C_hat, dtype=float).reshape(-1, 1)
    v = np.asarray(v_ray, dtype=float).reshape(-1, 1)
    num = float(v.T @ F @ C)
    den = float(v.T @ F @ v)
    if den <= 0:
        raise ValueError("Non-positive v^T F v. v may lie in a null direction or F is ill-conditioned.")
    chi2 = float(C.T @ F @ C - (num*num)/den)
    return chi2, num, den

if v_ray is None:
    print("Set v_ray (length N) in the previous cell and re-run.")
else:
    chi2, num, den = chi2_perp_from_F(C_hat, F, v_ray)
    nu_eff = max(rankF - 1, 0)
    print("chi2_perp =", chi2)
    print("rank(F)   =", rankF)
    print("nu_eff    =", nu_eff)
    print("v^T F C   =", num)
    print("v^T F v   =", den)

## 11) Save extracted arrays for download

In [ ]:
from pathlib import Path
np.save("coeff_names.npy", np.array(coeff_names, dtype=object))
np.save("A.npy", A)
np.save("Vinv.npy", Vinv)
np.save("F.npy", F)
np.save("SigmaC_pinv.npy", SigmaC)
np.save("C_hat.npy", C_hat)

print("Saved .npy files in:", Path(".").resolve())
!ls -lh *.npy

## 12) Download files (Colab)

Uncomment to download the saved arrays.

In [ ]:
# from google.colab import files
# for fn in ["coeff_names.npy","A.npy","Vinv.npy","F.npy","SigmaC_pinv.npy","C_hat.npy"]:
#     files.download(fn)